[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C21_Frontier_Pretraining_Course/02_tokenizer/02_tokenizer.ipynb)

# 02 · Tokenizer 设计（从零训练 BPE）

目标：用纯 numpy/stdlib 从零实现 **BPE 训练与编码**，算 **fertility / 压缩率**，画出词表大小的边际收益曲线，对比数字切分策略。

路线：字符基线 → BPE 训练(核心) → BPE 编码(对拍可逆) → fertility/压缩率 → 词表大小甜点曲线 → 数字按位切 → ✏️ 练习(bpe merge / fertility / 词表覆盖率 / 数字切分) → 📖 答案 → 🧪 真实 GPT-2 词表算账胶囊。

> 心智模型：**BPE 训练 = 反复数相邻对频率、合并最高频对；编码 = 按合并表顺序贪心粘合；fertility/压缩率 = 衡量切得多碎/多省**。

In [ ]:
import numpy as np
from collections import Counter, defaultdict
rng = np.random.default_rng(0)
print('环境就绪，开始训 tokenizer')

## 1 · 字符基线：最细的切分

先看最细的 baseline：按字符切。它零 OOV、词表小(就是字符集)，但序列最长、压缩率最低。BPE 要在它之上学合并来变短。

In [ ]:
text = ('the quick brown fox jumps over the lazy dog. '
        'the lazy dog sleeps. the quick fox runs. ' * 3)
n_bytes = len(text.encode('utf-8'))
char_tokens = list(text)
char_vocab = set(char_tokens)
print(f'原文 {n_bytes} 字节')
print(f'字符切分: {len(char_tokens)} tokens, 词表 {len(char_vocab)} 个字符')
print(f'压缩率 {n_bytes/len(char_tokens):.2f} 字节/token (≈1，最差)')
assert len(char_vocab) < 40, '字符词表很小'
print('✅ 字符基线：零 OOV 但序列最长 —— BPE 学合并把高频组合压短')

## 2 · 训练 BPE：反复合并最高频相邻对（本模块核心）

**核心算法**：把每个词表示成字符序列(末尾加 `</w>` 标词尾)；反复统计所有相邻对的频率、合并最高频对、记录合并规则，直到达到目标合并数。

In [ ]:
def get_word_freqs(text):
    '''词频统计。每个词表示成字符元组 + 词尾标记 </w>。'''
    freqs = Counter(text.split())
    return {tuple(list(w) + ['</w>']): c for w, c in freqs.items()}

def count_pairs(word_freqs):
    '''统计所有相邻 token 对的加权频率。'''
    pairs = Counter()
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += freq
    return pairs

def merge_pair(word_freqs, pair):
    '''把 word_freqs 中所有出现的 pair 合并成一个新 token。'''
    a, b = pair
    new_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i+1] == b:
                new_word.append(a + b)   # 合并
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_freqs[tuple(new_word)] = freq
    return new_freqs

def train_bpe(text, num_merges):
    '''训练 BPE，返回有序合并表 merges。'''
    word_freqs = get_word_freqs(text)
    merges = []
    for _ in range(num_merges):
        pairs = count_pairs(word_freqs)
        if not pairs:
            break
        # 取最高频对；平局时按字典序(确定性，可复现)
        best = max(pairs.items(), key=lambda kv: (kv[1], kv[0]))[0]
        word_freqs = merge_pair(word_freqs, best)
        merges.append(best)
    return merges

merges = train_bpe(text, num_merges=20)
print('前 8 条合并规则:')
for i, m in enumerate(merges[:8]):
    print(f'  #{i+1}: {m[0]!r} + {m[1]!r} -> {(m[0]+m[1])!r}')
assert len(merges) == 20
assert all(isinstance(m, tuple) and len(m) == 2 for m in merges)
# 高频词 'the' 的字符应较早被合并
merged_strs = [a+b for a,b in merges]
assert any('th' in s or 'the' in s for s in merged_strs), 'the 很高频，th 应被合并'
print('\n✅ BPE 训练：反复合并最高频对，产出有序合并表')

## 3 · BPE 编码：按合并表顺序贪心粘合（对拍可逆）

编码新文本：先拆成字符，再**严格按合并表顺序**反复合并。解码就是拼回去。验证 **encode→decode 可逆**(无损)。

In [ ]:
def bpe_encode_word(word, merges):
    '''对单个词编码：拆成字符+</w>，按 merges 顺序贪心合并。'''
    tokens = list(word) + ['</w>']
    for (a, b) in merges:                  # 严格按训练顺序！
        i = 0
        new = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new.append(a + b); i += 2
            else:
                new.append(tokens[i]); i += 1
        tokens = new
    return tokens

def bpe_encode(text, merges):
    out = []
    for w in text.split():
        out.extend(bpe_encode_word(w, merges))
    return out

def bpe_decode(tokens):
    '''拼回字符串：去掉 </w> 换成空格。'''
    s = ''.join(tokens)
    return s.replace('</w>', ' ').strip()

sample = 'the quick fox'
enc = bpe_encode(sample, merges)
dec = bpe_decode(enc)
print(f'原文:   {sample!r}')
print(f'编码:   {enc}')
print(f'解码:   {dec!r}')
assert dec == sample, 'encode->decode 必须可逆(无损)'
# 'the' 高频，应被压成很少的 token(理想是 1-2 个)
the_tokens = bpe_encode_word('the', merges)
print(f"\n'the' 编码成 {len(the_tokens)} 个 token: {the_tokens}")
assert len(the_tokens) < 4, 'the 很高频，应被压短'
print('✅ BPE 编码可逆，且把高频词压短')

## 4 · fertility 与压缩率：衡量 tokenizer 效率

**fertility** = token 数 ÷ 词数(每词切成几块)；**压缩率** = 字节数 ÷ token 数(每 token 顶几字节)。对比字符 vs BPE。

In [ ]:
def fertility(text, merges):
    n_words = len(text.split())
    n_tokens = len(bpe_encode(text, merges))
    return n_tokens / n_words

def compression_ratio(text, merges):
    n_bytes = len(text.encode('utf-8'))
    n_tokens = len(bpe_encode(text, merges))
    return n_bytes / n_tokens

# 字符基线(等价于 0 次合并)
fert_char = len(list(text.replace(' ',''))) / len(text.split())
print(f'字符切分:  fertility≈{fert_char:.2f} tok/词')
for nm in [10, 20, 50]:
    m = train_bpe(text, nm)
    print(f'BPE {nm:2d} merges: fertility {fertility(text, m):.2f} tok/词, 压缩率 {compression_ratio(text, m):.2f} 字节/tok')

m50 = train_bpe(text, 50)
m10 = train_bpe(text, 10)
assert fertility(text, m50) < fertility(text, m10), '更多合并 -> fertility 更低(更省)'
assert compression_ratio(text, m50) > compression_ratio(text, m10), '更多合并 -> 压缩率更高'
print('\n✅ 合并越多，fertility 越低、压缩率越高 —— 但下一节会看到收益递减')

## 5 · 词表大小的甜点曲线：压缩率上升但终将饱和

压缩率随合并数(≈词表大小)单调上升，但**终将饱和**——高频组合被合并完后，再加的合并都是长尾、边际增益回落。这条曲线决定了「词表多大划算」。

我们造一个**词汇丰富、Zipf 分布**的语料(才能看出真实的饱和)，扫描合并数画压缩率曲线。

In [ ]:
# 造一个词汇丰富、Zipf 频率的语料(小语料会过早饱和，看不出曲线)
corpus_rng = np.random.default_rng(7)
letters = list('abcdefghij')
pseudo_vocab = list(dict.fromkeys(
    ''.join(corpus_rng.choice(letters, size=corpus_rng.integers(3, 9))) for _ in range(400)))
zipf_w = 1.0 / np.arange(1, len(pseudo_vocab) + 1)
zipf_w /= zipf_w.sum()
big_text = ' '.join(corpus_rng.choice(pseudo_vocab, size=6000, p=zipf_w))
print(f'语料: {len(set(big_text.split()))} 个不同词, {len(big_text.split())} 总词, {len(big_text.encode())} 字节\n')

merge_counts = [0, 30, 80, 160, 320, 640, 1200]
ratios = []
for nm in merge_counts:
    if nm == 0:
        n_tok = len(list(big_text.replace(' ', '')))
    else:
        n_tok = len(bpe_encode(big_text, train_bpe(big_text, nm)))
    ratios.append(len(big_text.encode()) / n_tok)

print(f"{'合并数':>6} {'压缩率':>8} {'边际增益':>10}")
for i, (nm, r) in enumerate(zip(merge_counts, ratios)):
    delta = '' if i == 0 else f'+{r - ratios[i-1]:.3f}'
    print(f'{nm:6d} {r:8.3f} {delta:>10}')

deltas = [ratios[i+1] - ratios[i] for i in range(len(ratios)-1)]
# 压缩率单调上升
assert all(ratios[i] < ratios[i+1] for i in range(len(ratios)-1)), '压缩率应随词表单调上升'
# 饱和/收益递减：最后一段的边际增益低于峰值边际增益(高频组合已被吃光)
assert deltas[-1] < max(deltas), '末段边际增益应低于峰值(收益开始递减 -> 甜点区成因)'
print('\n✅ 压缩率单调升但末段边际增益回落(饱和) —— 词表不是越大越好，存在甜点区')

## 6 · 数字按位切：让模型看到位值结构

若 '127' 和 '128' 是无关的两个 token，模型学不会它们只差 1。**数字按单个数位切**(LLaMA/GPT-4 做法)让模型看到位值。对比两种策略。

In [ ]:
import re
def split_digits(text):
    '''把每个数字串拆成单个数位(用空格隔开)，其余不变。'''
    return re.sub(r'\d', lambda m: ' ' + m.group() + ' ', text)

nums = 'the year 2024 has 365 days and 12 months'
print('原文:      ', nums)
print('按位切后:  ', split_digits(nums).split())

# 不按位切：每个数字串是一个 token，相近数字毫无关系
whole_number_tokens = set(re.findall(r'\d+', 'the model saw 127 128 129 1000 2000'))
print(f'\n不按位切: 127/128/129 是 {len(whole_number_tokens)} 个无关 token: {sorted(whole_number_tokens)}')

# 按位切：所有数字共享 10 个数位 token，127 和 128 共享 '1''2'
digit_tokens = set(c for c in '127128129' )
print(f'按位切:   所有数字只用 {len(set("0123456789"))} 个数位 token，127 和 128 共享前缀')

t127 = [c for c in '127']
t128 = [c for c in '128']
shared = len(set(t127) & set(t128))
assert shared >= 2, '按位切后 127 和 128 应共享数位 token(看到只差末位)'
print('✅ 数字按位切：相近数字共享数位 token，模型能看到位值结构(算术能力的基础)')

---
## ✏️ 练习 1：实现一步 BPE 合并

实现 `bpe_step(word_freqs)`：统计相邻对频率，找到最高频对(平局按字典序)，合并它，返回 `(新word_freqs, 被合并的对)`。

复用 `count_pairs` / `merge_pair`。

In [ ]:
def bpe_step(word_freqs):
    # TODO: pairs=count_pairs(...); best=最高频对(平局字典序);
    #       返回 (merge_pair(word_freqs,best), best)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
wf = get_word_freqs('low low low lower newest newest')
wf2, pair = bpe_step(wf)
print(f'第一步合并了: {pair[0]!r}+{pair[1]!r}')
# 'low' 出现 4 次(low×3+lower×1), (l,o) 应是最高频之一
all_tokens = set(t for w in wf2 for t in w)
assert pair[0] + pair[1] in all_tokens, '合并后的新 token 应出现在词表里'
# 合并应减少总 token 数
before = sum(len(w) for w in wf)
after = sum(len(w) for w in wf2)
assert after < before, '一次合并应减少总 token 数'
print('✅ 练习 1 通过：一步 BPE 合并正确')

## ✏️ 练习 2：实现 fertility

实现 `fertility_ex(words, merges)`：`words` 是词列表，返回平均每词被 BPE 切成多少 token。复用 `bpe_encode_word`。

In [ ]:
def fertility_ex(words, merges):
    # TODO: 对每个词 bpe_encode_word，累加 token 数；返回 总token / 词数
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
m = train_bpe(text, 30)
words = ['the', 'quick', 'fox']
f = fertility_ex(words, m)
print(f'fertility = {f:.2f} tok/词')
# 必须等于手工算的
manual = sum(len(bpe_encode_word(w, m)) for w in words) / len(words)
assert abs(f - manual) < 1e-9
assert f >= 1.0, 'fertility 至少为 1'
# 合并越多 fertility 越低
assert fertility_ex(words, train_bpe(text, 60)) <= fertility_ex(words, train_bpe(text, 5))
print('✅ 练习 2 通过：fertility 正确')

## ✏️ 练习 3：词表覆盖率

实现 `vocab_coverage(text, merges)`：用 BPE 编码 `text`，返回**实际用到的不同 token 占合并表能产生的潜在词表的比例**的一个简化版——
具体：返回 `(编码后不同 token 数, 编码后 token 总数, 不同/总数的比例)`。比例低说明少数 token 反复出现(高频主导)。

In [ ]:
def vocab_coverage(text, merges):
    # TODO: enc=bpe_encode; 返回 (len(set(enc)), len(enc), len(set(enc))/len(enc))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
m = train_bpe(big_text, 40)
uniq, total, ratio = vocab_coverage(big_text, m)
print(f'不同 token {uniq}, 总 token {total}, 比例 {ratio:.3f}')
assert uniq <= total
assert 0 < ratio <= 1
# 高频文本(big_text 是重复的)应有大量重复 token -> 比例较低
assert ratio < 0.5, '重复语料里少数 token 主导，比例应较低'
print('✅ 练习 3 通过：词表覆盖率反映高频 token 的主导程度')

## ✏️ 练习 4：数字切分策略对比

实现 `count_unique_number_tokens(numbers, by_digit)`：给一组数字串，
- `by_digit=False`：每个数字串是一个 token，返回不同 token 数；
- `by_digit=True`：按单个数位切，返回不同 token 数(应 ≤ 10)。

验证按位切大幅减少数字相关的词表占用。

In [ ]:
def count_unique_number_tokens(numbers, by_digit):
    # TODO: by_digit=False -> set(numbers) 的大小;
    #       by_digit=True  -> 所有数字串里出现的不同数位字符的数量
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
numbers = [str(x) for x in range(100, 200)]    # 100 个不同的三位数
whole = count_unique_number_tokens(numbers, by_digit=False)
bydigit = count_unique_number_tokens(numbers, by_digit=True)
print(f'整数切分: {whole} 个不同 token')
print(f'按位切分: {bydigit} 个不同 token')
assert whole == 100, '100 个数字 = 100 个 token'
assert bydigit <= 10, '按位切最多 10 个数位 token'
assert bydigit < whole, '按位切大幅省词表'
print('✅ 练习 4 通过：按位切让数字只占 ≤10 个词表位，且暴露位值结构')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bpe_step(word_freqs):
    pairs = count_pairs(word_freqs)
    best = max(pairs.items(), key=lambda kv: (kv[1], kv[0]))[0]
    return merge_pair(word_freqs, best), best

In [ ]:
# 练习 2 参考答案
def fertility_ex(words, merges):
    total = sum(len(bpe_encode_word(w, merges)) for w in words)
    return total / len(words)

In [ ]:
# 练习 3 参考答案
def vocab_coverage(text, merges):
    enc = bpe_encode(text, merges)
    return len(set(enc)), len(enc), len(set(enc)) / len(enc)

In [ ]:
# 练习 4 参考答案
def count_unique_number_tokens(numbers, by_digit):
    if not by_digit:
        return len(set(numbers))
    digits = set()
    for num in numbers:
        digits |= set(num)
    return len(digits)

---
## 🧪 真实数据胶囊：GPT-2 / GPT-4 词表与序列长度算账

用真实词表大小算一笔账：GPT-2 用 50257 词表，GPT-4(cl100k) 用 ~100k，近期模型(如 Llama-3)到 128k。更大词表 -> 压缩率更高 -> 同样 context 装更多文本，但 embedding 参数更多。算这个权衡。

In [ ]:
# 真实词表大小与近似英文压缩率(字节/token，量级)
tokenizers = {
    'GPT-2 (50k)':    (50257, 3.6),
    'GPT-4 cl100k':   (100277, 4.0),
    'Llama-3 (128k)': (128256, 4.2),
}
HIDDEN = 4096          # 假设的模型隐藏维度
context_bytes = 100_000   # 想塞进 context 的英文字节数

print(f"{'tokenizer':18s} {'词表':>8s} {'压缩率':>7s} {'context装下tok':>14s} {'emb参数(M)':>11s}")
for name, (V, cr) in tokenizers.items():
    tokens_needed = context_bytes / cr            # 装下这些字节要多少 token
    emb_params = 2 * V * HIDDEN / 1e6              # 输入+输出 embedding
    print(f'{name:18s} {V:8d} {cr:7.1f} {tokens_needed:14.0f} {emb_params:11.1f}')

# 更大词表: 压缩率更高(同内容更少 token) 但 embedding 更大
V_small, cr_small = tokenizers['GPT-2 (50k)']
V_big, cr_big = tokenizers['Llama-3 (128k)']
assert cr_big > cr_small, '更大词表压缩率更高'
assert 2*V_big*HIDDEN > 2*V_small*HIDDEN, '更大词表 embedding 参数更多'
tok_saved = context_bytes/cr_small - context_bytes/cr_big
print(f'\n128k 词表 vs 50k: 同样 100KB 文本省 {tok_saved:.0f} token(序列更短)，'
      f'但多 {2*(V_big-V_small)*HIDDEN/1e6:.0f}M embedding 参数')
print('✅ 胶囊：词表大小是压缩率(省序列)与 embedding 参数的权衡')

**🧪 胶囊练习**：实现 `tokens_for_context(byte_budget, compression_ratio)`：给定字节预算与压缩率，返回需要多少 token。

In [ ]:
def tokens_for_context(byte_budget, compression_ratio):
    # TODO: 字节数 / 压缩率 = token 数
    raise NotImplementedError

In [ ]:
# 自测
t = tokens_for_context(100_000, 4.0)
print(f'100KB 文本，压缩率 4.0 -> 需要 {t:.0f} token')
assert abs(t - 25000) < 1
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def tokens_for_context(byte_budget, compression_ratio):
    return byte_budget / compression_ratio

### 小结
- **BPE**：从字符反复合并最高频相邻对，产出有序合并表；编码按合并表顺序贪心粘合，无损可逆。
- **三角权衡**：词表大→fertility 低/压缩率高(序列短)，但 embedding 参数多、token 稀疏、softmax 贵；压缩率边际递减 → 存在甜点区。
- **难处理的**：数字按位切(暴露位值，利算术)、代码合并缩进、多语 fertility 不公平(结构性，传导到成本与能力)。
- **三条路线**：BPE(频率合并)、WordPiece(似然合并)、Unigram(EM 剪枝+最大似然切分)；SentencePiece 统一实现，多语标准。

下一站：**模块 03 · μP 与超参迁移** —— 词表定了、数据干净了，怎么不在大模型上烧钱调超参？在小模型调好直接迁过去。